# Load `encounters_100k.csv` into `raw.encounters`

Reads the CSV with pandas, connects to SQL Server with pyodbc, and inserts all rows into **`raw.encounters`**.

- **Before you start:** run `01_create_raw_encounters.sql` once so the table exists.
- **Kernel:** select the `.venv-dbt-s1` Python environment.
- **Credentials:** the same environment variables dbt uses (`DBT_SQLSERVER_HOST`, `DBT_SQLSERVER_USER`, `DBT_SQLSERVER_PASSWORD`).
- **Re-runnable:** the table is emptied first and the load runs as one transaction, so you get either all 50,000 rows or none.

In [1]:
import os
import time

import pandas as pd
import pyodbc

CSV_PATH = r"C:\Users\waleed\Documents\dbt_S1\encounters_100k.csv"
SCHEMA = "raw"
TABLE = "encounters"
CHUNK_SIZE = 5000  # rows sent per round trip

## 1. Read the CSV
Every column is read as text (`dtype=str`) because the raw table is all `VARCHAR`; the dbt staging model assigns the real data types later.
Only truly empty cells become NULL. Text such as `"NA"` is kept as-is.

In [2]:
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, na_values=[""])

print(df.shape)  # expect (50000, 93)
df.head()

(50000, 93)


,patient_id,birth_year,age,sex,race_ethnicity,state,encounter_id,icd10_code,diagnosis_display,diagnosis_category,...,kit_lead_time_days,kit_expiration_date,stockout_risk_flag,weekly_procedure_volume,projected_demand_4wk,days_of_supply,hedis_hba1c_tested,hedis_hba1c_poor_control,pdsa_cycle_id,himss_emram_stage
0,PAT-1000000,1964,60,female,Black,NC,bdd640fb-0667-4ad1-9c80-317fa3b1799d,K35.80,Unspecified acute appendicitis,GI,...,3,2024-09-21,0,10,40,15.4,1,0,NaN,7
1,PAT-1000001,1986,38,male,Asian,MI,448aaa9e-66b2-4c5b-90c1-87fcce177b4e,L03.311,Cellulitis of abdominal wall,DERM,...,2,2025-10-18,1,5,20,7.0,0,0,NaN,7
2,PAT-1000002,1983,41,male,White,MI,20de435d-2031-4750-840d-b9b4885f6e66,I63.9,"Cerebral infarction, unspecified",CV,...,14,2024-04-11,0,15,60,12.1,0,0,NaN,5
3,PAT-1000003,1963,61,male,Asian,NC,490617f2-747b-4dba-88fe-3ccdc8b8d9c6,N20.0,Calculus of kidney,GU,...,5,2025-02-11,1,7,28,9.0,1,0,NaN,5
4,PAT-1000004,1956,68,male,White,CA,57c700aa-b7b5-4ea7-b5eb-d32d9ad620ab,K35.80,Unspecified acute appendicitis,GI,...,3,2025-02-26,0,3,12,36.0,1,1,NaN,5


## 2. Connect to SQL Server

In [3]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={os.environ['DBT_SQLSERVER_HOST']},1433;"
    "DATABASE=healthcare_dbt_dw;"
    f"UID={os.environ['DBT_SQLSERVER_USER']};"
    f"PWD={os.environ['DBT_SQLSERVER_PASSWORD']};"
    "Encrypt=yes;TrustServerCertificate=yes;",
    autocommit=False,
)
cursor = conn.cursor()
cursor.fast_executemany = True  # send rows in bulk instead of one by one

print(cursor.execute("SELECT @@SERVERNAME, DB_NAME()").fetchone())

('SQLLAB', 'healthcare_dbt_dw')


## 3. Insert the data

In [4]:
columns = ", ".join(f"[{c}]" for c in df.columns)
placeholders = ", ".join("?" * len(df.columns))
insert_sql = f"INSERT INTO [{SCHEMA}].[{TABLE}] ({columns}) VALUES ({placeholders})"

# pandas uses NaN for blanks; SQL Server needs None (NULL)
rows = df.astype(object).where(df.notna(), None).values.tolist()

start = time.time()
try:
    cursor.execute(f"TRUNCATE TABLE [{SCHEMA}].[{TABLE}]")  # makes the notebook re-runnable
    for i in range(0, len(rows), CHUNK_SIZE):
        cursor.executemany(insert_sql, rows[i : i + CHUNK_SIZE])
        print(f"inserted {min(i + CHUNK_SIZE, len(rows)):,} / {len(rows):,} rows")
    conn.commit()
    print(f"Done in {time.time() - start:.0f} s")
except Exception:
    conn.rollback()  # nothing is half-loaded
    raise

inserted 5,000 / 50,000 rows
inserted 10,000 / 50,000 rows
inserted 15,000 / 50,000 rows
inserted 20,000 / 50,000 rows
inserted 25,000 / 50,000 rows
inserted 30,000 / 50,000 rows
inserted 35,000 / 50,000 rows
inserted 40,000 / 50,000 rows
inserted 45,000 / 50,000 rows
inserted 50,000 / 50,000 rows
Done in 42 s


## 4. Check the result
For the full set of checks, run `03_validate_raw_encounters.sql`; every row should say PASS.

In [5]:
row_count = cursor.execute(f"SELECT COUNT(*) FROM [{SCHEMA}].[{TABLE}]").fetchval()
print(f"rows in {SCHEMA}.{TABLE}: {row_count:,}  (CSV had {len(df):,})")

cursor.close()
conn.close()

rows in raw.encounters: 50,000  (CSV had 50,000)
